# Module 0: Market Data Access

## Learning Objectives

By completing this notebook, you will:
1. Connect to the EIA API and retrieve petroleum inventory data
2. Access USDA agricultural reports programmatically
3. Structure raw commodity data for analysis
4. Visualize commodity time series
5. Understand data quality and validation techniques

## Prerequisites

- Python basics (functions, dictionaries, lists)
- EIA API key (free from https://www.eia.gov/opendata/register.php)
- Completed environment setup from Module 0 Guide 3

## Estimated Time: 45-60 minutes

---

## 1. Setup and Configuration

First, we'll import necessary libraries and configure our API access. The EIA (Energy Information Administration) provides free access to US energy data—critical for oil and gas markets.

**Key Concept:** API keys should never be hardcoded. We use environment variables to keep credentials secure.

In [ ]:
# Standard library imports
import os
from datetime import datetime, timedelta
from typing import Dict, List, Optional

# Third-party imports
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Configuration
load_dotenv()  # Load environment variables from .env file
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# API credentials
EIA_API_KEY = os.getenv('EIA_API_KEY')

if not EIA_API_KEY:
    print("⚠️  WARNING: EIA_API_KEY not found!")
    print("   Set it in your .env file or as an environment variable.")
else:
    print("✅ API key loaded successfully")

## 2. Building an EIA Data Client

We'll create a reusable client class for fetching EIA data. This follows best practices:
- Encapsulation of API logic
- Error handling
- Response validation
- Easy to extend for other data sources

### 2.1 Understanding the EIA API v2

The EIA API v2 structure:
```
https://api.eia.gov/v2/{category}/data/
  ?api_key=YOUR_KEY
  &frequency=weekly
  &data[0]=value
  &facets[series][]=SERIES_ID
```

**What this means:**
- `category`: petroleum, natural-gas, coal, etc.
- `frequency`: weekly, monthly, annual
- `series`: specific data series (e.g., crude oil stocks)

In [ ]:
class EIAClient:
    """
    Client for accessing EIA API v2.
    
    Provides methods to fetch petroleum and natural gas data with
    automatic error handling and response validation.
    """
    
    BASE_URL = "https://api.eia.gov/v2"
    
    def __init__(self, api_key: str):
        """Initialize with API key."""
        self.api_key = api_key
        self.session = requests.Session()
        
    def fetch_series(self, 
                    category: str,
                    series_id: str, 
                    frequency: str = 'weekly',
                    num_periods: int = 52) -> pd.DataFrame:
        """
        Fetch time series data from EIA.
        
        Parameters:
        -----------
        category : str
            Data category (e.g., 'petroleum', 'natural-gas')
        series_id : str
            Series identifier (e.g., 'WCESTUS1' for crude stocks)
        frequency : str
            Data frequency ('weekly', 'monthly', 'annual')
        num_periods : int
            Number of periods to retrieve
            
        Returns:
        --------
        pd.DataFrame
            DataFrame with 'period', 'value', and 'series' columns
        """
        # Build request parameters
        endpoint = f"{self.BASE_URL}/{category}/data"
        
        params = {
            'api_key': self.api_key,
            'frequency': frequency,
            'data[0]': 'value',
            'facets[series][]': series_id,
            'sort[0][column]': 'period',
            'sort[0][direction]': 'desc',
            'offset': 0,
            'length': num_periods
        }
        
        # Make request
        try:
            response = self.session.get(endpoint, params=params)
            response.raise_for_status()
            
            data = response.json()
            
            # Validate response structure
            if 'response' not in data or 'data' not in data['response']:
                raise ValueError("Unexpected API response structure")
            
            # Convert to DataFrame
            df = pd.DataFrame(data['response']['data'])
            
            # Parse period as datetime
            df['period'] = pd.to_datetime(df['period'])
            df['value'] = pd.to_numeric(df['value'])
            
            # Sort by date ascending
            df = df.sort_values('period').reset_index(drop=True)
            
            return df[['period', 'value', 'series']]
            
        except requests.exceptions.RequestException as e:
            raise Exception(f"API request failed: {e}")
        except (KeyError, ValueError) as e:
            raise Exception(f"Data parsing failed: {e}")

# Create client instance
eia = EIAClient(EIA_API_KEY)
print("✅ EIA client initialized")

## 3. Fetching Crude Oil Inventory Data

Let's retrieve US commercial crude oil stocks—one of the most watched indicators in energy markets. Released every Wednesday at 10:30 AM ET, this data drives oil price movements.

**Series ID:** `WCESTUS1` = Weekly US Commercial Crude Oil Stocks (excluding SPR)
**Unit:** Thousand barrels
**Frequency:** Weekly

In [ ]:
# Fetch 1 year of crude oil inventory data
crude_stocks = eia.fetch_series(
    category='petroleum/sum/sndw',
    series_id='WCESTUS1',
    frequency='weekly',
    num_periods=52
)

# Display first and last few rows
print("Latest crude oil inventory data:")
print(crude_stocks.tail())
print(f"\nData spans: {crude_stocks['period'].min()} to {crude_stocks['period'].max()}")
print(f"Latest inventory: {crude_stocks['value'].iloc[-1]:,.0f} thousand barrels")

### 💡 Exercise 3.1: Calculate Inventory Change

**Task:** Add a column to the DataFrame showing the week-over-week change in crude oil inventories.

**Hints:**
- Use pandas `.diff()` method
- Positive values = inventory build, negative = draw
- EIA reports this as "change from prior week"

**Expected Output:** DataFrame with new 'weekly_change' column

In [ ]:
# YOUR CODE HERE
# ---------------
# Calculate weekly change in inventories



# Display results


In [ ]:
# SOLUTION (for instructors)
# ---------------------------
crude_stocks['weekly_change'] = crude_stocks['value'].diff()

# Display recent changes
print("Recent weekly changes:")
print(crude_stocks[['period', 'value', 'weekly_change']].tail(10))

# Summary statistics
print(f"\nAverage weekly change: {crude_stocks['weekly_change'].mean():,.0f} thousand barrels")
print(f"Largest build: {crude_stocks['weekly_change'].max():,.0f} thousand barrels")
print(f"Largest draw: {crude_stocks['weekly_change'].min():,.0f} thousand barrels")

In [ ]:
# Auto-graded test for Exercise 3.1
def test_exercise_3_1():
    """Validate inventory change calculation."""
    assert 'weekly_change' in crude_stocks.columns, "❌ Column 'weekly_change' not found. Did you add it to the DataFrame?"
    
    assert crude_stocks['weekly_change'].notna().sum() > 0, "❌ weekly_change column is all NaN. Check your calculation."
    
    # Check first value is NaN (no prior week)
    assert pd.isna(crude_stocks['weekly_change'].iloc[0]), "❌ First weekly_change should be NaN (no prior week to compare to)"
    
    # Verify calculation is correct
    expected = crude_stocks['value'].iloc[5] - crude_stocks['value'].iloc[4]
    actual = crude_stocks['weekly_change'].iloc[5]
    assert abs(expected - actual) < 0.01, f"❌ Calculation error. Expected {expected}, got {actual}"
    
    print("✅ All tests passed! Inventory change calculated correctly.")

# Run tests
test_exercise_3_1()

## 4. Visualizing Commodity Time Series

Time series visualization reveals patterns, seasonality, and trends. For commodity traders, charts often matter more than tables.

**What to look for:**
- Seasonal patterns (winter gas demand, spring refinery maintenance)
- Trend direction (building or drawing)
- Range bounds (5-year average provides context)

In [ ]:
# Create visualization of crude oil stocks
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Absolute inventory levels
ax1.plot(crude_stocks['period'], crude_stocks['value'], linewidth=2, color='darkblue', label='US Crude Stocks')
ax1.axhline(y=crude_stocks['value'].mean(), color='red', linestyle='--', alpha=0.7, label='52-week average')
ax1.fill_between(crude_stocks['period'], 
                  crude_stocks['value'].quantile(0.25), 
                  crude_stocks['value'].quantile(0.75),
                  alpha=0.2, color='blue', label='25th-75th percentile')

ax1.set_title('US Commercial Crude Oil Inventories', fontsize=14, fontweight='bold')
ax1.set_ylabel('Thousand Barrels', fontsize=12)
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Plot 2: Weekly changes (builds/draws)
colors = ['green' if x > 0 else 'red' for x in crude_stocks['weekly_change'].fillna(0)]
ax2.bar(crude_stocks['period'], crude_stocks['weekly_change'], color=colors, alpha=0.6, width=5)
ax2.axhline(y=0, color='black', linewidth=1)

ax2.set_title('Weekly Inventory Changes (Builds/Draws)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Change (Thousand Barrels)', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Chart interpretation:")
print(f"  - Current level vs. 52-week average: {((crude_stocks['value'].iloc[-1] / crude_stocks['value'].mean() - 1) * 100):+.1f}%")
print(f"  - Weeks with builds: {(crude_stocks['weekly_change'] > 0).sum()}")
print(f"  - Weeks with draws: {(crude_stocks['weekly_change'] < 0).sum()}")

## 5. Fetching Multiple Series

Commodity analysis often requires comparing multiple related series. Let's fetch crude, gasoline, and distillate inventories together.

**Key Series:**
- `WCESTUS1`: Crude oil stocks
- `WGTSTUS1`: Gasoline stocks  
- `WDISTUS1`: Distillate fuel oil stocks

In [ ]:
# Define series to fetch
PETROLEUM_SERIES = {
    'crude_oil': 'WCESTUS1',
    'gasoline': 'WGTSTUS1',
    'distillate': 'WDISTUS1'
}

# Fetch all series
inventory_data = {}

for name, series_id in PETROLEUM_SERIES.items():
    print(f"Fetching {name}...")
    df = eia.fetch_series(
        category='petroleum/sum/sndw',
        series_id=series_id,
        frequency='weekly',
        num_periods=52
    )
    inventory_data[name] = df

print("\n✅ All series fetched successfully")

# Display latest values
print("\nLatest inventory levels:")
for name, df in inventory_data.items():
    latest = df['value'].iloc[-1]
    print(f"  {name.title()}: {latest:,.0f} thousand barrels")

### 💡 Exercise 5.1: Multi-Series Comparison

**Task:** Create a normalized comparison chart showing all three inventory series on the same scale.

**Requirements:**
1. Normalize each series to start at 100
2. Plot all three on the same chart
3. Add legend and labels

**Hints:**
- Normalization formula: `(value / first_value) * 100`
- This shows relative changes, not absolute levels

In [ ]:
# YOUR CODE HERE
# ---------------
# Create normalized comparison chart





In [ ]:
# SOLUTION
# --------
plt.figure(figsize=(14, 7))

for name, df in inventory_data.items():
    # Normalize to index (first value = 100)
    normalized = (df['value'] / df['value'].iloc[0]) * 100
    plt.plot(df['period'], normalized, linewidth=2, label=name.replace('_', ' ').title())

plt.axhline(y=100, color='black', linestyle='--', alpha=0.5, label='Starting level')
plt.title('Petroleum Inventories - Normalized Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Index (First Value = 100)', fontsize=12)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📊 Normalized view shows relative changes across products")

## 6. Data Quality and Validation

Real-world data has issues: missing values, outliers, revisions. Always validate before using for trading decisions.

**Common Data Issues:**
1. Missing values (holidays, reporting delays)
2. Outliers (data errors or genuine extreme events)
3. Revisions (EIA updates historical values)
4. Gaps in time series

In [ ]:
def validate_time_series(df: pd.DataFrame, series_name: str) -> dict:
    """
    Perform quality checks on commodity time series data.
    
    Returns dictionary with validation results.
    """
    results = {
        'series': series_name,
        'total_records': len(df),
        'date_range': (df['period'].min(), df['period'].max()),
        'issues': []
    }
    
    # Check for missing values
    missing = df['value'].isna().sum()
    if missing > 0:
        results['issues'].append(f"⚠️  {missing} missing values ({missing/len(df)*100:.1f}%)")
    
    # Check for zero or negative values (impossible for stocks)
    invalid = (df['value'] <= 0).sum()
    if invalid > 0:
        results['issues'].append(f"❌ {invalid} zero/negative values (data error)")
    
    # Check for outliers using IQR method
    Q1 = df['value'].quantile(0.25)
    Q3 = df['value'].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df['value'] < (Q1 - 3*IQR)) | (df['value'] > (Q3 + 3*IQR))).sum()
    if outliers > 0:
        results['issues'].append(f"⚠️  {outliers} potential outliers detected")
    
    # Check for time gaps (weekly data should have ~7 day intervals)
    df_sorted = df.sort_values('period').reset_index(drop=True)
    time_diffs = df_sorted['period'].diff().dt.days
    gaps = (time_diffs > 14).sum()  # More than 2 weeks is a gap
    if gaps > 0:
        results['issues'].append(f"⚠️  {gaps} time gaps (>14 days between records)")
    
    # Check for suspicious changes (>30% week-over-week)
    pct_change = df_sorted['value'].pct_change().abs()
    large_changes = (pct_change > 0.30).sum()
    if large_changes > 0:
        results['issues'].append(f"⚠️  {large_changes} large weekly changes (>30%)")
    
    results['valid'] = len(results['issues']) == 0
    
    return results

# Validate our crude oil data
validation = validate_time_series(crude_stocks, 'US Crude Oil Stocks')

print(f"Validation Results for {validation['series']}")
print(f"Total records: {validation['total_records']}")
print(f"Date range: {validation['date_range'][0].date()} to {validation['date_range'][1].date()}")
print(f"\nData quality: {'✅ CLEAN' if validation['valid'] else '⚠️  ISSUES FOUND'}")

if validation['issues']:
    print("\nIssues detected:")
    for issue in validation['issues']:
        print(f"  {issue}")
else:
    print("\nNo issues detected - data is clean and ready for analysis")

### 💡 Exercise 6.1: Build a Data Quality Dashboard

**Task:** Validate all three petroleum inventory series and create a summary report.

**Requirements:**
1. Run validation on crude, gasoline, and distillate
2. Create a summary DataFrame showing validation results
3. Flag any series with issues

**Expected Output:** DataFrame with columns: series, total_records, issues_count, status

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
validation_results = []

for name, df in inventory_data.items():
    validation = validate_time_series(df, name)
    validation_results.append({
        'series': name,
        'total_records': validation['total_records'],
        'issues_count': len(validation['issues']),
        'status': '✅ Clean' if validation['valid'] else '⚠️  Issues'
    })

# Create summary DataFrame
summary = pd.DataFrame(validation_results)
print("\nData Quality Summary:")
print(summary.to_string(index=False))

# Overall assessment
clean_count = (summary['issues_count'] == 0).sum()
print(f"\nOverall: {clean_count}/{len(summary)} series are clean")

In [ ]:
# Auto-graded test for Exercise 6.1
def test_exercise_6_1():
    """Validate data quality dashboard."""
    assert 'summary' in globals(), "❌ Variable 'summary' not found. Did you create the summary DataFrame?"
    assert isinstance(summary, pd.DataFrame), "❌ summary should be a DataFrame"
    assert len(summary) == 3, f"❌ Expected 3 rows (one per series), got {len(summary)}"
    assert 'status' in summary.columns, "❌ summary should have 'status' column"
    assert 'issues_count' in summary.columns, "❌ summary should have 'issues_count' column"
    
    print("✅ All tests passed! Data quality dashboard created correctly.")

test_exercise_6_1()

## 7. Exporting Data for Further Analysis

Once we've fetched and validated data, we typically want to save it for use in other analyses or trading systems.

**Common formats:**
- CSV: Human-readable, universally compatible
- Parquet: Compressed, fast for pandas
- JSON: For API integration

In [ ]:
# Create data directory if it doesn't exist
import os
os.makedirs('data', exist_ok=True)

# Export each series
for name, df in inventory_data.items():
    # CSV export
    csv_path = f'data/{name}_inventory.csv'
    df.to_csv(csv_path, index=False)
    print(f"✅ Saved {csv_path}")

# Also save combined dataset
combined = pd.DataFrame()
for name, df in inventory_data.items():
    df_copy = df.copy()
    df_copy['commodity'] = name
    combined = pd.concat([combined, df_copy], ignore_index=True)

combined.to_csv('data/all_inventories.csv', index=False)
print("\n✅ Saved combined dataset: data/all_inventories.csv")
print(f"   Total records: {len(combined)}")

## Summary

### Key Takeaways

1. **API Access is Critical** - The EIA provides free, reliable commodity data essential for trading

2. **Data Validation Matters** - Always check for missing values, outliers, and time gaps before analysis

3. **Visualization Reveals Patterns** - Charts show trends and seasonality that tables hide

4. **Reusable Code Saves Time** - Building client classes and validation functions pays off quickly

5. **Context is King** - Absolute values matter less than changes and comparisons to averages

### Skills Gained

✅ Connecting to commodity data APIs  
✅ Fetching and structuring time series data  
✅ Calculating period-over-period changes  
✅ Creating commodity market visualizations  
✅ Validating data quality  
✅ Exporting data for further analysis  

### What's Next

In the next notebook (`02_llm_basics.ipynb`), we'll use LLMs to extract insights from the commodity reports that accompany this data—learning to convert unstructured text into structured signals.

### Additional Resources

- **EIA API Documentation:** https://www.eia.gov/opendata/documentation.php
- **EIA Data Browser:** https://www.eia.gov/opendata/browser/ (explore available series)
- **pandas Time Series Guide:** https://pandas.pydata.org/docs/user_guide/timeseries.html
- **Commodity Data Best Practices:** Focus on consistency, validation, and version control

---

**Next:** [02_llm_basics.ipynb](./02_llm_basics.ipynb) - Introduction to LLM-Based Report Analysis